# 🧠 Brain Tumor Detection — Computer Vision Pipeline
> **AI Engineer Notebook** | Classification: Glioma · Meningioma · Pituitary · No Tumor

---
## Pipeline Overview
1. Environment Setup
2. Dataset Exploration & EDA
3. Data Preprocessing & Augmentation
4. Model Architecture (Transfer Learning — EfficientNetB0)
5. Training with Callbacks
6. Evaluation & Metrics
7. Grad-CAM Visualization
8. Inference on New Images

## 1️⃣ Environment Setup

In [ ]:
# Install dependencies
!pip install tensorflow opencv-python matplotlib seaborn scikit-learn Pillow -q

In [ ]:
import os, cv2, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from collections import Counter
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {tf.config.list_physical_devices("GPU")}')

## 2️⃣ Dataset Exploration & EDA

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
BASE_DIR   = Path('brain_tumor_data')
TRAIN_DIR  = BASE_DIR / 'Training'
TEST_DIR   = BASE_DIR / 'Testing'
CLASSES    = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 30

print('Classes found:', CLASSES)

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
def count_images(root: Path):
    return {cls: len(list((root/cls).glob('*.jpg')) +
                      list((root/cls).glob('*.jpeg')) +
                      list((root/cls).glob('*.png')))
            for cls in CLASSES}

train_counts = count_images(TRAIN_DIR)
test_counts  = count_images(TEST_DIR)

df_dist = pd.DataFrame({'Train': train_counts, 'Test': test_counts})
print(df_dist)
print(f'\nTotal train: {sum(train_counts.values())} | Total test: {sum(test_counts.values())}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (split, counts) in zip(axes, [('Train', train_counts), ('Test', test_counts)]):
    bars = ax.bar(counts.keys(), counts.values(),
                  color=['#E63946','#457B9D','#2A9D8F','#E9C46A'])
    ax.set_title(f'{split} Distribution', fontsize=13, fontweight='bold')
    ax.set_ylabel('Image Count')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(int(bar.get_height())), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('eda_distribution.png', dpi=150)
plt.show()

In [ ]:
# ── Sample images per class ───────────────────────────────────────────────────
fig, axes = plt.subplots(len(CLASSES), 5, figsize=(14, 3*len(CLASSES)))
for r, cls in enumerate(CLASSES):
    imgs = list((TRAIN_DIR/cls).glob('*.jpg'))[:5]
    for c, img_path in enumerate(imgs):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        axes[r, c].imshow(img)
        axes[r, c].axis('off')
        if c == 0:
            axes[r, c].set_ylabel(cls, fontsize=12, fontweight='bold', rotation=0,
                                   labelpad=60, va='center')
plt.suptitle('Sample Images per Class', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_samples.png', dpi=150)
plt.show()

## 3️⃣ Data Preprocessing & Augmentation

In [ ]:
# ── ImageDataGenerators ───────────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    validation_split=0.15
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', seed=42
)
val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', seed=42
)
test_gen = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

CLASS_INDICES = {v:k for k,v in train_gen.class_indices.items()}
NUM_CLASSES   = len(CLASSES)
print('Class indices:', CLASS_INDICES)

## 4️⃣ Model Architecture — EfficientNetB0 (Transfer Learning)

In [ ]:
def build_model(num_classes: int, img_size=(224,224,3)) -> Model:
    """
    Transfer Learning dengan EfficientNetB0.
    Phase 1 : hanya classifier yang dilatih (base frozen)
    Phase 2 : fine-tune 30 layer terakhir
    """
    base = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=img_size
    )
    base.trainable = False  # Phase 1 - freeze base

    inputs  = keras.Input(shape=img_size, name='input_image')
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D(name='gap')(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(256, activation='relu', name='fc1')(x)
    x       = layers.Dropout(0.4, name='dropout1')(x)
    x       = layers.Dense(128, activation='relu', name='fc2')(x)
    x       = layers.Dropout(0.3, name='dropout2')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs, outputs, name='BrainTumorDetector')
    return model, base

model, base_model = build_model(NUM_CLASSES)
model.summary()

## 5️⃣ Training

In [ ]:
# ── Phase 1: Train classifier head ───────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase1 = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_model_phase1.h5', save_best_only=True, verbose=1)
]

print('=== Phase 1: Training Classifier Head ===')
hist1 = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=callbacks_phase1
)

In [ ]:
# ── Phase 2: Fine-tune last 30 layers ─────────────────────────────────────────
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase2 = [
    EarlyStopping(patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_model.h5', save_best_only=True, verbose=1)
]

print('=== Phase 2: Fine-Tuning ===')
hist2 = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks_phase2
)

In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────────
def plot_history(h1, h2):
    acc  = h1.history['accuracy']  + h2.history['accuracy']
    vacc = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss = h1.history['loss']      + h2.history['loss']
    vloss= h1.history['val_loss']  + h2.history['val_loss']
    ep   = range(1, len(acc)+1)
    split_ep = len(h1.history['accuracy'])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    for ax, (tr, va, title) in zip([ax1,ax2],
        [(acc,vacc,'Accuracy'),(loss,vloss,'Loss')]):
        ax.plot(ep, tr,  label='Train',      color='#2A9D8F', lw=2)
        ax.plot(ep, va,  label='Validation', color='#E63946', lw=2, ls='--')
        ax.axvline(split_ep, color='#457B9D', ls=':', lw=1.5, label='Phase 2 start')
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.legend(); ax.set_xlabel('Epoch')
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150)
    plt.show()

plot_history(hist1, hist2)

## 6️⃣ Evaluation & Metrics

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_gen, verbose=1)
print(f'\nTest Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Test Loss     : {test_loss:.4f}')

In [ ]:
# ── Confusion Matrix & Classification Report ──────────────────────────────────
test_gen.reset()
y_pred_prob = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = test_gen.classes

print('\n── Classification Report ──')
print(classification_report(y_true, y_pred, target_names=CLASSES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, linewidths=0.5)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 7️⃣ Grad-CAM Visualization

In [ ]:
def make_gradcam(model, img_array, last_conv='top_conv'):
    """Compute Grad-CAM heatmap for the predicted class."""
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        pred_idx  = tf.argmax(preds[0])
        pred_score = preds[:, pred_idx]

    grads  = tape.gradient(pred_score, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap).numpy()
    heatmap = np.maximum(heatmap, 0) / (np.max(heatmap) + 1e-8)
    return heatmap, int(pred_idx), float(preds[0][pred_idx])

def overlay_gradcam(img_path, model, img_size=(224,224)):
    orig  = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    img   = cv2.resize(orig, img_size) / 255.0
    inp   = np.expand_dims(img, 0).astype('float32')

    heatmap, pred_idx, confidence = make_gradcam(model, inp)
    heatmap_r = cv2.resize(heatmap, (orig.shape[1], orig.shape[0]))
    heatmap_c = np.uint8(255 * heatmap_r)
    heatmap_c = cv2.applyColorMap(heatmap_c, cv2.COLORMAP_JET)
    heatmap_c = cv2.cvtColor(heatmap_c, cv2.COLOR_BGR2RGB)
    overlay   = cv2.addWeighted(orig, 0.6, heatmap_c, 0.4, 0)

    return orig, overlay, CLASS_INDICES[pred_idx], confidence


# ── Show Grad-CAM for 1 sample per class ──────────────────────────────────────
fig, axes = plt.subplots(len(CLASSES), 2, figsize=(10, 4*len(CLASSES)))
for r, cls in enumerate(CLASSES):
    sample = next((TEST_DIR/cls).glob('*.jpg'))
    orig, overlay, pred_cls, conf = overlay_gradcam(sample, model)

    axes[r, 0].imshow(orig);     axes[r, 0].set_title(f'Original | True: {cls}')
    axes[r, 1].imshow(overlay);  axes[r, 1].set_title(f'Grad-CAM | Pred: {pred_cls} ({conf:.1%})')
    for ax in axes[r]: ax.axis('off')

plt.suptitle('Grad-CAM Explanations', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('gradcam_results.png', dpi=150)
plt.show()

## 8️⃣ Inference on New Images

In [ ]:
def predict_single(img_path: str, model, img_size=(224,224)):
    """
    Predict class for a single MRI image.
    Returns: predicted class, confidence, all class probabilities.
    """
    img = Image.open(img_path).convert('RGB').resize(img_size)
    arr = np.expand_dims(np.array(img) / 255.0, 0).astype('float32')

    probs    = model.predict(arr, verbose=0)[0]
    pred_idx = np.argmax(probs)
    label    = CLASS_INDICES[pred_idx]
    conf     = probs[pred_idx]

    print(f'\n🔬 Prediction  : {label.upper()}')
    print(f'   Confidence  : {conf:.2%}')
    print('   Probabilities:')
    for i, (c, p) in enumerate(zip(CLASSES, probs)):
        bar = '█' * int(p * 30)
        print(f'   {c:12s} {bar:<30} {p:.2%}')

    # ── Visualize ──────────────────────────────────────────────────────────
    _, overlay, _, _ = overlay_gradcam(img_path, model)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))
    ax1.imshow(np.array(Image.open(img_path).convert('RGB')))
    ax1.set_title('Input MRI', fontweight='bold'); ax1.axis('off')

    ax2.imshow(overlay)
    ax2.set_title(f'Grad-CAM | {label} ({conf:.1%})', fontweight='bold'); ax2.axis('off')

    colors = ['#E63946','#2A9D8F','#457B9D','#E9C46A']
    ax3.barh(CLASSES, probs, color=colors)
    ax3.set_xlim(0, 1)
    ax3.set_title('Class Probabilities', fontweight='bold')
    ax3.set_xlabel('Confidence')
    for i, p in enumerate(probs):
        ax3.text(p + 0.01, i, f'{p:.1%}', va='center', fontsize=10)

    plt.tight_layout()
    plt.show()
    return label, conf


# ── Example usage ─────────────────────────────────────────────────────────────
# Ganti path di bawah dengan gambar MRI baru
SAMPLE_PATH = str(next((TEST_DIR / CLASSES[0]).glob('*.jpg')))
predict_single(SAMPLE_PATH, model)

---
## ✅ Summary

| Step | Detail |
|------|--------|
| Dataset | Brain Tumor MRI — 4 kelas |
| Model | EfficientNetB0 + Custom Head |
| Strategy | Transfer Learning (2-phase) |
| Augmentation | Rotation, Shift, Zoom, Flip, Brightness |
| Explainability | Grad-CAM Heatmap |
| Output | `best_model.h5` + confusion matrix + training curves |